# Treemün 2.0.0 benchmark: surrogate functions versus original biomass tables

This notebook evaluates whether the growth surrogate functions used by Treemün reproduce the original tabulated dry-biomass values **and**, more importantly, whether both biomass representations lead to similar forest-management decisions.

The benchmark has four levels:

1. **Curve fidelity:** source-table values versus surrogate predictions at the original observed ages.
2. **Trajectory fidelity:** annual standing and harvested dry biomass for every stand-policy alternative.
3. **Decision fidelity:** comparison of the binary policy-selection variables \(x_{ij}\).
4. **Outcome fidelity:** comparison of the annual harvested-biomass variable \(H_t\), objective values, and cross-evaluated regret.

## Main experimental design

The primary optimization benchmark uses a **pure NPV objective** with:

- no carbon objective or carbon constraint;
- no adjacency or green-up constraints;
- no even-flow constraint;
- no minimum ending-stock requirement;
- no terminal value.

This isolates the effect of replacing the original source tables with the surrogate growth functions. Carbon is available only as an optional **ex post diagnostic** and never changes the selected decisions in the main experiment.

Because this unconstrained NPV model contains only the single-policy-per-stand constraints, it is separable by stand. The notebook therefore computes the exact analytical optimum and, when a MILP solver is available, verifies it with the Treemün Pyomo model.


## Scientific unit convention

This notebook follows the manuscript-wide convention:

- dry aboveground biomass per unit area: **Mg ha\(^{-1}\)**;
- stand- or landscape-level dry aboveground biomass: **Mg**;
- standing-wood carbon: **Mg C**;
- carbon stock-time: **Mg C yr**.

One megagram is exactly one metric tonne:

\[
1~\mathrm{Mg}=1~\text{metric tonne}.
\]

Treemün retains some historical machine-readable API names such as
`harvested_dry_wood_t`, `standing_dry_wood_t_before_operation`, and
`carbon_stock_time_tC_year`. These names are preserved only for package
compatibility. Their values are interpreted and reported here as Mg,
Mg C, or Mg C yr, respectively. No numerical conversion is required.


## Methodological note on the table-based growth engine

The original tables contain discrete ages, whereas Treemün evaluates biomass at arbitrary integer ages required by the management policies. The table engine uses:

- exact values at observed ages;
- monotone PCHIP interpolation between observed ages;
- a zero-biomass anchor at age 0 for ages below the first reported value;
- linear continuation using the last observed slope above the table support.

Every table evaluation is labelled as `observed`, `interpolated`, `extrapolated_low`, or `extrapolated_high`. These labels are exported so that conclusions can be separated according to the degree of direct table support.

For Pinus, the same segmentation used by the original R fitting script is reproduced: rows are divided at `Po/Ra` and `Raleo` events, with the second segment representing the Treemün pre-thinning curve and the third segment representing the post-thinning curve. For pulpwood, the post-raleo segment is used for the unthinned Treemün equation, matching the original fitting workflow.

In [ ]:
from __future__ import annotations

from dataclasses import dataclass
from pathlib import Path
from typing import Mapping, Sequence
import os
import sys
import time
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from scipy.interpolate import PchipInterpolator
from scipy.stats import spearmanr

import treemun_sim as tm
from treemun_sim.simulation import (
    build_policy_catalog,
    is_policy_feasible,
    kitral_class,
)

pd.set_option("display.max_columns", 100)
pd.set_option("display.width", 180)

print("Treemün version:", tm.__version__)
print("Treemün location:", Path(tm.__file__).resolve())

In [ ]:

UNIT_MG = "Mg"
UNIT_MG_HA = r"Mg ha$^{-1}$"
UNIT_MG_C = "Mg C"
UNIT_MG_C_YR = "Mg C yr"

TREEMUN_PUBLICATION_COLUMN_ALIASES = {
    "harvested_dry_wood_t": "harvested_dry_wood_Mg",
    "standing_dry_wood_t_before_operation": (
        "standing_dry_wood_Mg_before_operation"
    ),
    "standing_dry_wood_t_after_operation": (
        "standing_dry_wood_Mg_after_operation"
    ),
    "ending_dry_wood_t": "ending_dry_wood_Mg",
    "carbon_stock_time_tC_year": "carbon_stock_time_Mg_C_yr",
    "standing_wood_carbon_tC_after_operation": (
        "standing_wood_carbon_Mg_C_after_operation"
    ),
}


def publication_table(frame: pd.DataFrame) -> pd.DataFrame:
    """Return a copy with publication-facing Mg-based column names."""
    renamed_columns = {}
    for column in frame.columns:
        publication_name = str(column)
        for old, new in TREEMUN_PUBLICATION_COLUMN_ALIASES.items():
            publication_name = publication_name.replace(old, new)
        renamed_columns[column] = publication_name
    return frame.rename(columns=renamed_columns).copy()


print(
    "Scientific units:",
    UNIT_MG_HA,
    "for per-hectare biomass;",
    UNIT_MG,
    "for stand/landscape biomass;",
    UNIT_MG_C_YR,
    "for carbon stock-time.",
)


## 1. Configuration and file paths

In [ ]:
# -----------------------------
# Files
# -----------------------------
# Place the source XLSM/CSV beside this notebook, or edit SOURCE_TABLE_FILE.

WORKING_DIRECTORY = Path.cwd().resolve()


def first_existing(candidates: Sequence[Path], label: str) -> Path:
    for candidate in candidates:
        candidate = candidate.expanduser().resolve()
        if candidate.exists():
            return candidate
    formatted = "\n".join(f"  - {candidate}" for candidate in candidates)
    raise FileNotFoundError(f"Could not locate {label}. Checked:\n{formatted}")


SOURCE_TABLE_FILE = first_existing(
    [
        WORKING_DIRECTORY / "benchmark_notebooks_and_generator/DatosArmonizados_Plantaciones_VFinal.xlsm",
        WORKING_DIRECTORY.parent / "benchmark_notebooks_and_generator/DatosArmonizados_Plantaciones_VFinal.xlsm",
        WORKING_DIRECTORY / "DatosArmonizados_Plantaciones_VFinal.xlsm",
        WORKING_DIRECTORY / "benchmark_notebooks_and_generator/DatosArmonizados_Plantaciones_VFinal.csv",
        WORKING_DIRECTORY.parent / "benchmark_notebooks_and_generator/DatosArmonizados_Plantaciones_VFinal.csv",
        WORKING_DIRECTORY / "DatosArmonizados_Plantaciones_VFinal.csv",
    ],
    "original harmonized biomass source table",
)

PACKAGE_DIRECTORY = Path(tm.__file__).resolve().parent
LOOKUP_FILE = PACKAGE_DIRECTORY / "data" / "lookup_table.csv"

STANDS_FILE = first_existing(
    [
        WORKING_DIRECTORY / "forest_stands.csv",
        WORKING_DIRECTORY / "examples" / "forest_stands.csv",
        WORKING_DIRECTORY.parent / "examples" / "forest_stands.csv",
        PACKAGE_DIRECTORY.parent / "examples" / "forest_stands.csv",
    ],
    "105-stand input file",
)

OUTPUT_DIRECTORY = WORKING_DIRECTORY / "outputs_surrogate_table_benchmark"
OUTPUT_DIRECTORY.mkdir(parents=True, exist_ok=True)

print("Source table:", SOURCE_TABLE_FILE)
print("Lookup table:", LOOKUP_FILE)
print("Stand table:", STANDS_FILE)
print("Output directory:", OUTPUT_DIRECTORY)

In [ ]:
# -----------------------------
# Benchmark settings
# -----------------------------

HORIZON = 30
DISCOUNT_RATE = 0.08
FALLBACK_THINNING_FRACTION = 0.30
MINIMUM_CURVE_RESIDUAL_FRACTION = 0.10

# Exact comparison is desirable because small NPV differences can change x_ij.
RELATIVE_GAP = 0.0
SOLVER_THREAD_CAP = 8
SOLVER_TIME_LIMIT_SECONDS = None
PREFERRED_SOLVER = "cplex"
SOLVER_CANDIDATES = (
    "cplex",
    "appsi_cplex",
    "highs",
    "appsi_highs",
    "cbc",
)

RUN_MILP_VERIFICATION = True
RUN_OPTIONAL_CARBON_DIAGNOSTIC = False
SAVE_ALL_CURVE_PLOTS = False

# Constant 2025 USD reference parameters for Chile.
# Terminal value is intentionally zero in this benchmark.
CHILE_2025_NPV_PARAMETERS = {
    "revenue_per_dry_t": {
        tm.PINUS: 44.07,
        tm.EUCALYPTUS: 43.66,
    },
    "variable_cost_per_dry_t": {
        tm.PINUS: 0.0,
        tm.EUCALYPTUS: 0.0,
    },
    "transport_cost_per_dry_t": {
        tm.PINUS: 15.76,
        tm.EUCALYPTUS: 10.59,
    },
    "thinning_cost_per_dry_t": {
        tm.PINUS: 11.03,
        tm.EUCALYPTUS: 0.0,
    },
    "final_harvest_cost_per_dry_t": {
        tm.PINUS: 12.61,
        tm.EUCALYPTUS: 8.47,
    },
    "thinning_cost_per_ha": 0.0,
    "final_harvest_cost_per_ha": 0.0,
    "replanting_cost_per_ha": {
        tm.PINUS: 684.0,
        tm.EUCALYPTUS: 716.0,
    },
    "annual_management_cost_per_ha": 15.0,
    "replanting_delay_periods": 0,
    "terminal_value_per_dry_t": 0.0,
}

print("Horizon:", HORIZON)
print("Discount rate:", f"{DISCOUNT_RATE:.1%}")
print("Terminal value included: no")
print("Carbon/adjacency constraints included: no")

## 2. Load the Treemün lookup and 105-stand landscape

In [ ]:
lookup = tm.load_lookup_table(LOOKUP_FILE)
stands = tm.load_stand_table(STANDS_FILE)

print("Lookup equations:", len(lookup))
print("Stands:", len(stands))
print("Total area (ha):", f"{stands['area_ha'].sum():,.3f}")

display(lookup.head())
display(stands.head())

## 3. Load and harmonize the original source tables

In [ ]:
SOURCE_COLUMNS = [
    "ID",
    "Especie",
    "Zona",
    "SI",
    "Manejo",
    "Codigo",
    "Tipo_de_manejo",
    "Manejo_especifico",
    "Edad",
    "N_arboles",
    "Area_basal",
    "Altura_dominante",
    "DAP_dominante",
    "Biomasa_fustal_total",
    "Biomasa_fustal_IU10",
    "Biomasa_fustal_IU15",
    "Biomasa_fustal_IU20",
    "Biomasa_fustal_IU30",
    "Volumen_IU0",
    "Volumen_IU8",
    "Volumen_IU16",
    "Volumen_IU24",
    "Volumen_IU32",
    "Biomasa_fustal_hasta_IU0",
    "Biomasa_fustal_hasta_IU8",
    "Biomasa_fustal_hasta_IU16",
    "Biomasa_fustal_hasta_IU24",
    "Biomasa_fustal_hasta_IU32",
    "Biomasa_fustal_sobre_IU8",
    "Biomasa_fustal_sobre_IU16",
    "Biomasa_fustal_sobre_IU24",
    "Biomasa_fustal_sobre_IU32",
    "Biomasa_fustal_component_IU10",
    "Biomasa_fustal_component_IU15",
    "Biomasa_fustal_component_IU20",
    "Biomasa_fustal_component_IU30",
    "Total_fustal",
    "Hojas",
    "Ramas",
    "Corteza",
    "Total_no_fustal",
    "Total",
]


def load_original_biomass_table(path: Path) -> pd.DataFrame:
    """Load the harmonized source sheet and retain rows with total dry biomass."""
    suffix = path.suffix.lower()
    if suffix in {".xlsm", ".xlsx"}:
        raw = pd.read_excel(
            path,
            sheet_name="Armonización",
            header=None,
            skiprows=4,
            engine="openpyxl",
        )
    elif suffix == ".csv":
        # The source CSV contains a multi-line header. The first omitted data row
        # has no Total value, so dropping incomplete Total values preserves the
        # fitting observations used by the original R script.
        raw = pd.read_csv(
            path,
            sep=";",
            encoding="latin1",
            header=None,
            skiprows=5,
            engine="python",
        )
    else:
        raise ValueError(f"Unsupported source-table format: {path.suffix}")

    raw = raw.iloc[:, :42].copy()
    raw.columns = SOURCE_COLUMNS
    raw = raw.replace({"NA": np.nan, "–": np.nan, "-": np.nan, "": np.nan})

    numeric_columns = [
        "ID", "Zona", "SI", "Manejo", "Edad", "N_arboles", "Total"
    ]
    for column in numeric_columns:
        raw[column] = pd.to_numeric(raw[column], errors="coerce")

    for column in ["Especie", "Tipo_de_manejo", "Manejo_especifico"]:
        raw[column] = (
            raw[column]
            .astype("string")
            .str.replace("\u00a0", " ", regex=False)
            .str.strip()
        )

    raw["source_row_order"] = np.arange(len(raw))
    raw = raw.loc[raw["Total"].notna()].copy()
    raw["species"] = raw["Especie"].replace(
        {
            "Pinues radiata": tm.PINUS,
            "Pinus radiata": tm.PINUS,
            "Eucalyptus": tm.EUCALYPTUS,
            "Eucalyptus globulus": tm.EUCALYPTUS,
        }
    )

    unsupported = raw.loc[~raw["species"].isin([tm.PINUS, tm.EUCALYPTUS])]
    if not unsupported.empty:
        raise ValueError(
            "Unsupported species labels in source data: "
            f"{unsupported['Especie'].dropna().unique().tolist()}"
        )
    return raw.reset_index(drop=True)


source_data = load_original_biomass_table(SOURCE_TABLE_FILE)

print("Source observations with Total dry biomass:", len(source_data))
print(source_data.groupby(["species", "Zona"])["Total"].count())
display(source_data.head())

## 4. Reconstruct the exact source segments used by the surrogate equations

In [ ]:
MANAGEMENT_LABELS = {
    "intensive_1": "Intensivo",
    "intensive_2": "Intensivo2",
    "multipurpose": "Multipropósito",
    "pulpwood": "Pulpable",
}


def split_pinus_management_segments(group: pd.DataFrame) -> list[pd.DataFrame]:
    """Replicate the segment construction used in ModeloBiomasaRodal.r."""
    ordered = group.sort_values("source_row_order").reset_index(drop=True)
    event_positions = np.flatnonzero(
        ordered["Manejo_especifico"].isin(["Raleo", "Po/Ra"]).to_numpy()
    )
    bounds = [-1, *event_positions.tolist(), len(ordered) - 1]
    return [
        ordered.iloc[left + 1 : right + 1].copy()
        for left, right in zip(bounds[:-1], bounds[1:])
    ]


def build_source_curve_catalog(
    source: pd.DataFrame,
    lookup_table: pd.DataFrame,
) -> pd.DataFrame:
    """Map every Treemün equation ID to its original tabulated observations."""
    records: list[dict[str, object]] = []

    for equation in lookup_table.itertuples(index=False):
        if equation.species == tm.EUCALYPTUS:
            subset = source.loc[
                source["species"].eq(tm.EUCALYPTUS)
                & source["Zona"].eq(int(equation.zone))
                & source["SI"].eq(int(equation.site_index))
            ].copy()
            if int(equation.initial_density_trees_ha) == 800:
                subset = subset.loc[subset["N_arboles"].le(800)]
            else:
                subset = subset.loc[subset["N_arboles"].gt(800)]
            source_segment = "unthinned_source_table"

        else:
            source_management = MANAGEMENT_LABELS[str(equation.management_regime)]
            complete_management_table = source.loc[
                source["species"].eq(tm.PINUS)
                & source["Zona"].eq(int(equation.zone))
                & source["SI"].eq(int(equation.site_index))
                & source["Tipo_de_manejo"].eq(source_management)
            ].copy()

            segments = split_pinus_management_segments(complete_management_table)

            if str(equation.growth_curve).startswith("pre_thinning"):
                subset = segments[1]
                source_segment = "pre_thinning_source_segment"
            elif str(equation.growth_curve).startswith("post_thinning"):
                subset = segments[2]
                source_segment = "post_thinning_source_segment"
            elif (
                str(equation.growth_curve) == "unthinned"
                and str(equation.management_regime) == "pulpwood"
            ):
                subset = segments[1]
                source_segment = "pulpwood_source_segment"
            else:
                raise ValueError(
                    "Could not map equation to source segment: "
                    f"equation_id={equation.equation_id}"
                )

        points = (
            subset[["Edad", "Total"]]
            .dropna()
            .groupby("Edad", as_index=False)["Total"]
            .mean()
            .sort_values("Edad")
        )
        if len(points) < 2:
            raise ValueError(
                f"Equation {equation.equation_id} has fewer than two source points."
            )

        for point in points.itertuples(index=False):
            records.append(
                {
                    "equation_id": int(equation.equation_id),
                    "species": str(equation.species),
                    "zone": int(equation.zone),
                    "site_index": int(equation.site_index),
                    "management_regime": str(equation.management_regime),
                    "growth_curve": str(equation.growth_curve),
                    "initial_density_trees_ha": int(
                        equation.initial_density_trees_ha
                    ),
                    "source_segment": source_segment,
                    "age": float(point.Edad),
                    "source_dry_biomass_Mg_ha": float(point.Total),
                }
            )

    catalog = pd.DataFrame.from_records(records)
    mapped_ids = set(catalog["equation_id"].unique())
    expected_ids = set(lookup_table["equation_id"].astype(int))
    if mapped_ids != expected_ids:
        raise AssertionError(
            f"Source mapping mismatch. Missing={expected_ids-mapped_ids}; "
            f"extra={mapped_ids-expected_ids}"
        )
    return catalog


source_curve_catalog = build_source_curve_catalog(source_data, lookup)

curve_support_summary = (
    source_curve_catalog.groupby(
        [
            "equation_id",
            "species",
            "zone",
            "site_index",
            "management_regime",
            "growth_curve",
        ],
        as_index=False,
    )
    .agg(
        source_points=("age", "size"),
        minimum_source_age=("age", "min"),
        maximum_source_age=("age", "max"),
        minimum_source_biomass=("source_dry_biomass_Mg_ha", "min"),
        maximum_source_biomass=("source_dry_biomass_Mg_ha", "max"),
    )
)

print("Mapped equation IDs:", source_curve_catalog["equation_id"].nunique())
print("Mapped source points:", len(source_curve_catalog))
display(curve_support_summary)

## 5. Curve-level fidelity metrics

In [ ]:
def regression_metrics(observed: np.ndarray, predicted: np.ndarray) -> dict[str, float]:
    observed = np.asarray(observed, dtype=float)
    predicted = np.asarray(predicted, dtype=float)
    error = predicted - observed
    absolute_error = np.abs(error)

    ss_res = float(np.sum(error**2))
    ss_tot = float(np.sum((observed - observed.mean()) ** 2))
    r_squared = np.nan if ss_tot <= 0 else 1.0 - ss_res / ss_tot

    nonzero = np.abs(observed) > 1e-12
    mape = (
        float(np.mean(absolute_error[nonzero] / np.abs(observed[nonzero])) * 100)
        if nonzero.any()
        else np.nan
    )
    smape_denominator = np.abs(observed) + np.abs(predicted)
    smape_mask = smape_denominator > 1e-12
    smape = (
        float(
            np.mean(
                2.0 * absolute_error[smape_mask] / smape_denominator[smape_mask]
            )
            * 100
        )
        if smape_mask.any()
        else np.nan
    )

    observed_range = float(observed.max() - observed.min())
    rmse = float(np.sqrt(np.mean(error**2)))

    return {
        "n": int(len(observed)),
        "r_squared": r_squared,
        "rmse_Mg_ha": rmse,
        "mae_Mg_ha": float(np.mean(absolute_error)),
        "bias_Mg_ha": float(np.mean(error)),
        "mape_percent": mape,
        "smape_percent": smape,
        "maximum_absolute_error_Mg_ha": float(absolute_error.max()),
        "normalized_rmse_percent": (
            np.nan if observed_range <= 0 else 100.0 * rmse / observed_range
        ),
    }


curve_comparison = source_curve_catalog.merge(
    lookup[
        [
            "equation_id",
            "next_equation_id",
            "alpha",
            "beta",
            "gamma",
        ]
    ],
    on="equation_id",
    how="left",
    validate="many_to_one",
)
curve_comparison["surrogate_dry_biomass_Mg_ha"] = np.maximum(
    curve_comparison["alpha"]
    * curve_comparison["age"].pow(curve_comparison["beta"])
    + curve_comparison["gamma"],
    0.0,
)
curve_comparison["residual_Mg_ha"] = (
    curve_comparison["surrogate_dry_biomass_Mg_ha"]
    - curve_comparison["source_dry_biomass_Mg_ha"]
)
curve_comparison["absolute_error_Mg_ha"] = curve_comparison[
    "residual_Mg_ha"
].abs()

metric_records = []
metadata_columns = [
    "equation_id",
    "species",
    "zone",
    "site_index",
    "management_regime",
    "growth_curve",
    "initial_density_trees_ha",
    "source_segment",
]
for keys, group in curve_comparison.groupby(metadata_columns, dropna=False):
    record = dict(zip(metadata_columns, keys))
    record.update(
        regression_metrics(
            group["source_dry_biomass_Mg_ha"].to_numpy(),
            group["surrogate_dry_biomass_Mg_ha"].to_numpy(),
        )
    )
    metric_records.append(record)

curve_metrics = pd.DataFrame(metric_records).sort_values(
    ["species", "equation_id"]
)

species_curve_metrics = []
for species, group in curve_comparison.groupby("species"):
    record = {"species": species}
    record.update(
        regression_metrics(
            group["source_dry_biomass_Mg_ha"].to_numpy(),
            group["surrogate_dry_biomass_Mg_ha"].to_numpy(),
        )
    )
    species_curve_metrics.append(record)
species_curve_metrics = pd.DataFrame(species_curve_metrics)

display(species_curve_metrics)
display(curve_metrics)

curve_comparison.to_csv(
    OUTPUT_DIRECTORY / "curve_point_comparison.csv", index=False
)
curve_metrics.to_csv(
    OUTPUT_DIRECTORY / "curve_metrics_by_equation.csv", index=False
)
species_curve_metrics.to_csv(
    OUTPUT_DIRECTORY / "curve_metrics_by_species.csv", index=False
)

In [ ]:
# Representative source-table points and surrogate curves.
REPRESENTATIVE_EQUATION_IDS = [8, 13, 21, 23, 26, 28, 29, 31]
representative_ids = [
    equation_id
    for equation_id in REPRESENTATIVE_EQUATION_IDS
    if equation_id in set(curve_comparison["equation_id"])
]

number_of_columns = 2
number_of_rows = int(np.ceil(len(representative_ids) / number_of_columns))
fig, axes = plt.subplots(
    number_of_rows,
    number_of_columns,
    figsize=(14, 4.2 * number_of_rows),
    squeeze=False,
)

for axis, equation_id in zip(axes.flat, representative_ids):
    group = curve_comparison.loc[
        curve_comparison["equation_id"].eq(equation_id)
    ].sort_values("age")
    metadata = group.iloc[0]
    dense_age = np.linspace(group["age"].min(), group["age"].max(), 250)
    dense_prediction = np.maximum(
        metadata["alpha"] * dense_age ** metadata["beta"] + metadata["gamma"],
        0.0,
    )
    axis.scatter(
        group["age"],
        group["source_dry_biomass_Mg_ha"],
        label="Original table",
    )
    axis.plot(dense_age, dense_prediction, label="Surrogate")
    axis.set_title(
        f"Eq. {equation_id}: {metadata['species']} | "
        f"Z{metadata['zone']} | SI {metadata['site_index']}\n"
        f"{metadata['management_regime']} | {metadata['growth_curve']}"
    )
    axis.set_xlabel("Biological age")
    axis.set_ylabel(r"Dry biomass (Mg ha$^{-1}$)")
    axis.grid(alpha=0.25)
    axis.legend()

for axis in axes.flat[len(representative_ids):]:
    axis.axis("off")

fig.tight_layout()
fig.savefig(
    OUTPUT_DIRECTORY / "representative_curve_comparison.png",
    dpi=220,
    bbox_inches="tight",
)
plt.show()

In [ ]:
fig, axis = plt.subplots(figsize=(10, 6))
for species, group in curve_comparison.groupby("species"):
    axis.scatter(
        group["source_dry_biomass_Mg_ha"],
        group["residual_Mg_ha"],
        label=species,
        alpha=0.7,
    )
axis.axhline(0.0, linewidth=1)
axis.set_xlabel("Original table dry biomass (Mg ha$^{-1}$)")
axis.set_ylabel("Surrogate minus table (Mg ha$^{-1}$)")
axis.set_title("Surrogate residuals at original tabulated ages")
axis.grid(alpha=0.25)
axis.legend()
fig.tight_layout()
fig.savefig(
    OUTPUT_DIRECTORY / "curve_residuals.png",
    dpi=220,
    bbox_inches="tight",
)
plt.show()

## 6. Build the table-based biomass evaluator

In [ ]:
@dataclass
class SourceTableCurve:
    equation_id: int
    observed_ages: np.ndarray
    observed_values: np.ndarray

    def __post_init__(self) -> None:
        order = np.argsort(self.observed_ages)
        self.observed_ages = np.asarray(self.observed_ages, dtype=float)[order]
        self.observed_values = np.asarray(self.observed_values, dtype=float)[order]

        if self.observed_ages[0] > 0:
            interpolation_ages = np.r_[0.0, self.observed_ages]
            interpolation_values = np.r_[0.0, self.observed_values]
        else:
            interpolation_ages = self.observed_ages.copy()
            interpolation_values = self.observed_values.copy()

        unique = np.r_[True, np.diff(interpolation_ages) > 0]
        self._interpolation_ages = interpolation_ages[unique]
        self._interpolation_values = interpolation_values[unique]
        self._interpolator = PchipInterpolator(
            self._interpolation_ages,
            self._interpolation_values,
            extrapolate=False,
        )

    @property
    def minimum_observed_age(self) -> float:
        return float(self.observed_ages.min())

    @property
    def maximum_observed_age(self) -> float:
        return float(self.observed_ages.max())

    def support_class(self, age: float) -> str:
        age = float(age)
        if np.any(np.isclose(age, self.observed_ages)):
            return "observed"
        if age < self.minimum_observed_age:
            return "extrapolated_low"
        if age > self.maximum_observed_age:
            return "extrapolated_high"
        return "interpolated"

    def dry_biomass_t_ha(self, age: float) -> float:
        age = float(age)
        if age < self._interpolation_ages[0]:
            value = self._interpolation_values[0]
        elif age <= self._interpolation_ages[-1]:
            value = float(self._interpolator(age))
        else:
            final_slope = (
                self._interpolation_values[-1]
                - self._interpolation_values[-2]
            ) / (
                self._interpolation_ages[-1]
                - self._interpolation_ages[-2]
            )
            value = self._interpolation_values[-1] + final_slope * (
                age - self._interpolation_ages[-1]
            )
        return max(float(value), 0.0)


def build_table_curve_providers(
    catalog: pd.DataFrame,
) -> dict[int, SourceTableCurve]:
    return {
        int(equation_id): SourceTableCurve(
            equation_id=int(equation_id),
            observed_ages=group["age"].to_numpy(),
            observed_values=group["source_dry_biomass_Mg_ha"].to_numpy(),
        )
        for equation_id, group in catalog.groupby("equation_id")
    }


table_curve_providers = build_table_curve_providers(source_curve_catalog)
assert set(table_curve_providers) == set(lookup["equation_id"].astype(int))

print("Table curve providers:", len(table_curve_providers))

## 7. Table-based trajectory simulator with the same Treemün management logic

In [ ]:
def lookup_equation_row(lookup_table: pd.DataFrame, equation_id: int) -> pd.Series:
    matches = lookup_table.loc[
        lookup_table["equation_id"].eq(int(equation_id))
    ]
    if len(matches) != 1:
        raise ValueError(f"Unknown or duplicate equation_id: {equation_id}")
    return matches.iloc[0]


def lookup_initial_equation_row(
    stand,
    lookup_table: pd.DataFrame,
) -> pd.Series:
    matches = lookup_table.loc[
        lookup_table["species"].eq(stand.species)
        & lookup_table["zone"].eq(int(stand.zone))
        & lookup_table["site_index"].eq(int(stand.site_index))
        & lookup_table["management_regime"].eq(stand.management_regime)
        & lookup_table["growth_curve"].eq(stand.growth_curve)
        & lookup_table["initial_density_trees_ha"].eq(
            int(stand.initial_density_trees_ha)
        )
    ]
    if len(matches) != 1:
        raise ValueError(
            f"Expected one initial equation for {stand.stand_id!r}; "
            f"found {len(matches)}."
        )
    return matches.iloc[0]


def simulate_table_based_forest(
    stands_table: pd.DataFrame,
    lookup_table: pd.DataFrame,
    curve_providers: Mapping[int, SourceTableCurve],
    *,
    horizon: int,
    pinus_policies=tm.DEFAULT_PINUS_POLICIES,
    eucalyptus_policies=tm.DEFAULT_EUCALYPTUS_POLICIES,
    fallback_thinning_fraction: float = 0.30,
    minimum_curve_residual_fraction: float = 0.10,
):
    """Simulate source-table trajectories using Treemün 2.0.0 operations."""
    policy_catalog = build_policy_catalog(pinus_policies, eucalyptus_policies)

    forest: list[pd.DataFrame] = []
    policy_summary: list[dict[str, object]] = []
    ending_stock: dict[tuple[str, str], float] = {}
    harvested_by_period: dict[tuple[int, str, str, str], float] = {}

    for stand in stands_table.itertuples(index=False):
        initial_equation = lookup_initial_equation_row(stand, lookup_table)
        post_thinning_equation = (
            lookup_equation_row(
                lookup_table,
                int(initial_equation["next_equation_id"]),
            )
            if pd.notna(initial_equation["next_equation_id"])
            else None
        )

        feasible_policy_count = 0
        for policy in policy_catalog[str(stand.species)]:
            if not is_policy_feasible(
                str(stand.species),
                int(stand.initial_age),
                int(horizon),
                policy,
            ):
                continue

            feasible_policy_count += 1
            age = int(stand.initial_age)
            already_thinned = False
            post_thinning_continuity_adjustment_t = 0.0
            force_initial_final_harvest = (
                str(stand.species) == tm.PINUS
                and policy.thinning_age is not None
                and age > policy.thinning_age
            )

            records: list[dict[str, object]] = []
            for period in range(1, int(horizon) + 1):
                before_equation = (
                    post_thinning_equation
                    if already_thinned and post_thinning_equation is not None
                    else initial_equation
                )
                before_curve = curve_providers[int(before_equation["equation_id"])]
                unadjusted_stock_t = (
                    before_curve.dry_biomass_t_ha(age) * float(stand.area_ha)
                )
                before_stock_t = max(
                    unadjusted_stock_t
                    + (
                        post_thinning_continuity_adjustment_t
                        if already_thinned
                        and post_thinning_equation is not None
                        and int(before_equation["equation_id"])
                        == int(post_thinning_equation["equation_id"])
                        else 0.0
                    ),
                    0.0,
                )

                operation = "none"
                harvested_t = 0.0
                after_stock_t = before_stock_t
                after_equation = before_equation

                thinning_method = "not_applicable"
                fallback_triggered = False
                candidate_post_t = np.nan
                candidate_residual_fraction = np.nan
                candidate_thinning_fraction = np.nan
                applied_thinning_fraction = np.nan
                candidate_post_support = "not_applicable"

                overdue_reset = period == 1 and force_initial_final_harvest
                scheduled_final_harvest = age == policy.final_harvest_age
                is_final_harvest = overdue_reset or scheduled_final_harvest
                is_thinning = (
                    str(stand.species) == tm.PINUS
                    and policy.thinning_age is not None
                    and age == policy.thinning_age
                    and not already_thinned
                    and not is_final_harvest
                )

                if is_final_harvest:
                    operation = "final_harvest"
                    harvested_t = before_stock_t
                    after_stock_t = 0.0
                    after_equation = initial_equation

                elif is_thinning:
                    operation = "thinning"
                    if post_thinning_equation is not None:
                        post_curve = curve_providers[
                            int(post_thinning_equation["equation_id"])
                        ]
                        candidate_post_t = (
                            post_curve.dry_biomass_t_ha(age)
                            * float(stand.area_ha)
                        )
                        candidate_post_support = post_curve.support_class(age)

                        if before_stock_t <= 0.0:
                            raise ValueError(
                                f"Non-positive pre-thinning stock for "
                                f"{stand.stand_id!r}, age={age}."
                            )
                        candidate_residual_fraction = (
                            candidate_post_t / before_stock_t
                        )
                        candidate_thinning_fraction = (
                            before_stock_t - candidate_post_t
                        ) / before_stock_t

                        if candidate_post_t > before_stock_t + 1e-8:
                            raise ValueError(
                                "Table post-thinning stock exceeds pre-thinning "
                                f"stock for {stand.stand_id!r}, age={age}: "
                                f"{candidate_post_t:.6f} > {before_stock_t:.6f}."
                            )

                        fallback_triggered = (
                            candidate_residual_fraction
                            <= float(minimum_curve_residual_fraction)
                        )
                        if fallback_triggered:
                            thinning_method = "fixed_fraction_fallback"
                            applied_thinning_fraction = float(
                                fallback_thinning_fraction
                            )
                            harvested_t = (
                                applied_thinning_fraction * before_stock_t
                            )
                            after_stock_t = before_stock_t - harvested_t
                            post_thinning_continuity_adjustment_t = (
                                after_stock_t - candidate_post_t
                            )
                        else:
                            thinning_method = "curve_difference"
                            harvested_t = before_stock_t - candidate_post_t
                            after_stock_t = candidate_post_t
                            applied_thinning_fraction = (
                                candidate_thinning_fraction
                            )
                            post_thinning_continuity_adjustment_t = 0.0

                        after_equation = post_thinning_equation
                    else:
                        thinning_method = "fixed_fraction_no_post_curve"
                        fallback_triggered = True
                        applied_thinning_fraction = float(
                            fallback_thinning_fraction
                        )
                        harvested_t = applied_thinning_fraction * before_stock_t
                        after_stock_t = before_stock_t - harvested_t
                        post_thinning_continuity_adjustment_t = 0.0
                    already_thinned = True

                managed_after_operation = already_thinned and not is_final_harvest
                records.append(
                    {
                        "period": period,
                        "stand_id": str(stand.stand_id),
                        "species": str(stand.species),
                        "area_ha": float(stand.area_ha),
                        "policy": policy.name,
                        "policy_number": policy.policy_number,
                        "stand_age": age,
                        "operation": operation,
                        "final_harvest_reason": (
                            "overdue_thinning_rotation_reset"
                            if overdue_reset
                            else (
                                "scheduled_final_harvest"
                                if scheduled_final_harvest
                                else "not_applicable"
                            )
                        ),
                        "initial_rotation_reset_triggered": bool(overdue_reset),
                        "management_state": (
                            "post_thinning"
                            if managed_after_operation
                            else "unthinned"
                        ),
                        "growth_curve_before_operation": str(
                            before_equation["growth_curve"]
                        ),
                        "growth_curve_after_operation": str(
                            after_equation["growth_curve"]
                        ),
                        "equation_id_before_operation": int(
                            before_equation["equation_id"]
                        ),
                        "equation_id_after_operation": int(
                            after_equation["equation_id"]
                        ),
                        "source_support_before_operation": (
                            before_curve.support_class(age)
                        ),
                        "source_support_candidate_post_thinning": (
                            candidate_post_support
                        ),
                        "unadjusted_curve_dry_wood_t_before_operation": round(
                            unadjusted_stock_t, 3
                        ),
                        "post_thinning_continuity_adjustment_t": round(
                            post_thinning_continuity_adjustment_t, 3
                        ),
                        "standing_dry_wood_t_before_operation": round(
                            before_stock_t, 3
                        ),
                        "harvested_dry_wood_t": round(harvested_t, 3),
                        "standing_dry_wood_t_after_operation": round(
                            after_stock_t, 3
                        ),
                        "thinning_calculation_method": thinning_method,
                        "thinning_fallback_triggered": fallback_triggered,
                        "candidate_post_thinning_dry_wood_t": (
                            round(float(candidate_post_t), 3)
                            if np.isfinite(candidate_post_t)
                            else np.nan
                        ),
                        "candidate_curve_residual_fraction": (
                            round(float(candidate_residual_fraction), 6)
                            if np.isfinite(candidate_residual_fraction)
                            else np.nan
                        ),
                        "candidate_thinning_fraction": (
                            round(float(candidate_thinning_fraction), 6)
                            if np.isfinite(candidate_thinning_fraction)
                            else np.nan
                        ),
                        "applied_thinning_fraction": (
                            round(float(applied_thinning_fraction), 6)
                            if np.isfinite(applied_thinning_fraction)
                            else np.nan
                        ),
                        "kitral_class": kitral_class(
                            str(stand.species), age, managed_after_operation
                        ),
                        "thinning_age": policy.thinning_age,
                        "final_harvest_age": policy.final_harvest_age,
                    }
                )

                if is_final_harvest:
                    age = 1
                    already_thinned = False
                    post_thinning_continuity_adjustment_t = 0.0
                else:
                    age += 1

            trajectory = pd.DataFrame.from_records(records)
            forest.append(trajectory)
            ending_stock[(str(stand.stand_id), policy.name)] = float(
                trajectory.iloc[-1]["standing_dry_wood_t_after_operation"]
            )
            for row in trajectory.itertuples(index=False):
                if row.harvested_dry_wood_t > 0:
                    harvested_by_period[
                        (row.period, row.species, row.policy, row.stand_id)
                    ] = float(row.harvested_dry_wood_t)

            policy_summary.append(
                {
                    "stand_id": str(stand.stand_id),
                    "species": str(stand.species),
                    "area_ha": float(stand.area_ha),
                    "initial_age": int(stand.initial_age),
                    "ending_age": int(trajectory.iloc[-1]["stand_age"]),
                    "policy": policy.name,
                    "policy_number": policy.policy_number,
                    "thinning_age": policy.thinning_age,
                    "final_harvest_age": policy.final_harvest_age,
                    "initial_equation_id": int(
                        trajectory.iloc[0]["equation_id_before_operation"]
                    ),
                }
            )

        if feasible_policy_count == 0:
            raise ValueError(
                f"Stand {stand.stand_id!r} has no feasible policy."
            )

    return forest, policy_summary, ending_stock, harvested_by_period

## 8. Generate surrogate-based and table-based trajectories

In [ ]:
simulation_started = time.perf_counter()

(
    surrogate_forest,
    surrogate_policy_summary,
    surrogate_ending_stock,
    surrogate_harvested_by_period,
) = tm.simulate_forest(
    stands_file=STANDS_FILE,
    horizon=HORIZON,
    include_carbon=False,
    fallback_thinning_fraction=FALLBACK_THINNING_FRACTION,
    minimum_curve_residual_fraction=MINIMUM_CURVE_RESIDUAL_FRACTION,
)

(
    table_forest,
    table_policy_summary,
    table_ending_stock,
    table_harvested_by_period,
) = simulate_table_based_forest(
    stands,
    lookup,
    table_curve_providers,
    horizon=HORIZON,
    fallback_thinning_fraction=FALLBACK_THINNING_FRACTION,
    minimum_curve_residual_fraction=MINIMUM_CURVE_RESIDUAL_FRACTION,
)

simulation_elapsed = time.perf_counter() - simulation_started

surrogate_keys = {
    (str(record["stand_id"]), str(record["policy"]))
    for record in surrogate_policy_summary
}
table_keys = {
    (str(record["stand_id"]), str(record["policy"]))
    for record in table_policy_summary
}
assert surrogate_keys == table_keys
assert len(surrogate_forest) == len(table_forest)

print("Common alternatives:", len(surrogate_keys))
print("Annual rows per representation:", sum(len(x) for x in surrogate_forest))
print(f"Total simulation time: {simulation_elapsed:,.2f} s")

In [ ]:
surrogate_long = pd.concat(surrogate_forest, ignore_index=True)
table_long = pd.concat(table_forest, ignore_index=True)

support_distribution = (
    table_long.groupby(
        ["operation", "source_support_before_operation"],
        as_index=False,
    )
    .size()
    .sort_values(["operation", "size"], ascending=[True, False])
)

table_harvest_support = (
    table_long.loc[table_long["harvested_dry_wood_t"].gt(0)]
    .groupby(
        [
            "operation",
            "final_harvest_reason",
            "source_support_before_operation",
        ],
        as_index=False,
    )
    .size()
)

print("Support classes for all annual evaluations:")
display(support_distribution)
print("Support classes at positive-harvest events:")
display(table_harvest_support)

support_distribution.to_csv(
    OUTPUT_DIRECTORY / "table_support_distribution.csv", index=False
)
table_harvest_support.to_csv(
    OUTPUT_DIRECTORY / "table_harvest_support.csv", index=False
)

## 9. Trajectory-level comparison for every stand-policy alternative

In [ ]:
trajectory_columns = [
    "stand_id",
    "policy",
    "period",
    "species",
    "stand_age",
    "operation",
    "standing_dry_wood_t_before_operation",
    "harvested_dry_wood_t",
    "standing_dry_wood_t_after_operation",
]

trajectory_comparison = surrogate_long[trajectory_columns].merge(
    table_long[
        trajectory_columns
        + [
            "source_support_before_operation",
            "source_support_candidate_post_thinning",
        ]
    ],
    on=["stand_id", "policy", "period", "species", "stand_age", "operation"],
    how="outer",
    suffixes=("_surrogate", "_table"),
    validate="one_to_one",
    indicator=True,
)

assert trajectory_comparison["_merge"].eq("both").all()

for variable in [
    "standing_dry_wood_t_before_operation",
    "harvested_dry_wood_t",
    "standing_dry_wood_t_after_operation",
]:
    trajectory_comparison[f"{variable}_difference"] = (
        trajectory_comparison[f"{variable}_surrogate"]
        - trajectory_comparison[f"{variable}_table"]
    )
    trajectory_comparison[f"{variable}_absolute_difference"] = trajectory_comparison[
        f"{variable}_difference"
    ].abs()

trajectory_metric_records = []
for (stand_id, policy, species), group in trajectory_comparison.groupby(
    ["stand_id", "policy", "species"]
):
    harvested_error = group["harvested_dry_wood_t_difference"].to_numpy()
    before_error = group[
        "standing_dry_wood_t_before_operation_difference"
    ].to_numpy()
    trajectory_metric_records.append(
        {
            "stand_id": stand_id,
            "policy": policy,
            "species": species,
            "annual_stock_rmse_Mg": float(np.sqrt(np.mean(before_error**2))),
            "annual_stock_mae_Mg": float(np.mean(np.abs(before_error))),
            "annual_harvest_rmse_Mg": float(
                np.sqrt(np.mean(harvested_error**2))
            ),
            "annual_harvest_mae_Mg": float(np.mean(np.abs(harvested_error))),
            "surrogate_total_harvest_Mg": float(
                group["harvested_dry_wood_t_surrogate"].sum()
            ),
            "table_total_harvest_Mg": float(
                group["harvested_dry_wood_t_table"].sum()
            ),
            "maximum_annual_harvest_difference_Mg": float(
                np.max(np.abs(harvested_error))
            ),
        }
    )

trajectory_metrics = pd.DataFrame(trajectory_metric_records)
trajectory_metrics["total_harvest_difference_Mg"] = (
    trajectory_metrics["surrogate_total_harvest_Mg"]
    - trajectory_metrics["table_total_harvest_Mg"]
)

trajectory_summary = (
    trajectory_metrics.groupby("species", as_index=False)
    .agg(
        alternatives=("policy", "size"),
        median_annual_stock_rmse_Mg=("annual_stock_rmse_Mg", "median"),
        maximum_annual_stock_rmse_Mg=("annual_stock_rmse_Mg", "max"),
        median_annual_harvest_rmse_Mg=("annual_harvest_rmse_Mg", "median"),
        maximum_annual_harvest_rmse_Mg=("annual_harvest_rmse_Mg", "max"),
        median_total_harvest_difference_Mg=(
            "total_harvest_difference_Mg",
            "median",
        ),
    )
)

display(trajectory_summary)
display(
    trajectory_metrics.sort_values(
        "annual_harvest_rmse_Mg", ascending=False
    ).head(15)
)

publication_table(trajectory_comparison).to_csv(
    OUTPUT_DIRECTORY / "annual_trajectory_point_comparison.csv",
    index=False,
)
trajectory_metrics.to_csv(
    OUTPUT_DIRECTORY / "trajectory_metrics_by_alternative.csv", index=False
)
trajectory_summary.to_csv(
    OUTPUT_DIRECTORY / "trajectory_metrics_by_species.csv", index=False
)

## 10. Evaluate the same NPV formulation for all alternatives

In [ ]:
surrogate_economics = tm.evaluate_forest_economics(
    surrogate_forest,
    economic_mode="detailed_cash_flow",
    discount_rate=DISCOUNT_RATE,
    economic_parameters=CHILE_2025_NPV_PARAMETERS,
)

table_economics = tm.evaluate_forest_economics(
    table_forest,
    economic_mode="detailed_cash_flow",
    discount_rate=DISCOUNT_RATE,
    economic_parameters=CHILE_2025_NPV_PARAMETERS,
)

surrogate_npv = surrogate_economics.summary_by_policy.rename(
    columns={"economic_value": "surrogate_npv"}
)
table_npv = table_economics.summary_by_policy.rename(
    columns={"economic_value": "table_npv"}
)

npv_comparison = surrogate_npv[
    [
        "stand_id",
        "policy",
        "species",
        "surrogate_npv",
        "discounted_gross_revenue",
        "discounted_total_cost",
    ]
].merge(
    table_npv[
        [
            "stand_id",
            "policy",
            "table_npv",
            "discounted_gross_revenue",
            "discounted_total_cost",
        ]
    ],
    on=["stand_id", "policy"],
    how="inner",
    suffixes=("_surrogate", "_table"),
    validate="one_to_one",
)

npv_comparison["npv_difference"] = (
    npv_comparison["surrogate_npv"] - npv_comparison["table_npv"]
)
npv_comparison["absolute_npv_difference"] = npv_comparison[
    "npv_difference"
].abs()
npv_comparison["relative_npv_difference_percent"] = np.where(
    npv_comparison["table_npv"].abs() > 1e-9,
    100.0
    * npv_comparison["npv_difference"]
    / npv_comparison["table_npv"].abs(),
    np.nan,
)

npv_metric_summary = (
    npv_comparison.groupby("species", as_index=False)
    .agg(
        alternatives=("policy", "size"),
        mean_npv_difference=("npv_difference", "mean"),
        median_absolute_npv_difference=("absolute_npv_difference", "median"),
        maximum_absolute_npv_difference=("absolute_npv_difference", "max"),
        median_relative_npv_difference_percent=(
            "relative_npv_difference_percent",
            "median",
        ),
    )
)

display(npv_metric_summary)
display(
    npv_comparison.sort_values("absolute_npv_difference", ascending=False).head(15)
)

npv_comparison.to_csv(
    OUTPUT_DIRECTORY / "npv_coefficients_by_alternative.csv", index=False
)
npv_metric_summary.to_csv(
    OUTPUT_DIRECTORY / "npv_difference_summary.csv", index=False
)

In [ ]:
# Ranking stability and near-tie diagnostics by stand.
ranking_records = []
for stand_id, group in npv_comparison.groupby("stand_id"):
    ordered_surrogate = group.sort_values(
        ["surrogate_npv", "policy"], ascending=[False, True]
    )
    ordered_table = group.sort_values(
        ["table_npv", "policy"], ascending=[False, True]
    )

    correlation = spearmanr(
        group["surrogate_npv"],
        group["table_npv"],
        nan_policy="omit",
    ).statistic

    surrogate_values = ordered_surrogate["surrogate_npv"].to_numpy()
    table_values = ordered_table["table_npv"].to_numpy()
    ranking_records.append(
        {
            "stand_id": stand_id,
            "species": group["species"].iloc[0],
            "spearman_policy_rank_correlation": float(correlation),
            "surrogate_best_policy": ordered_surrogate.iloc[0]["policy"],
            "table_best_policy": ordered_table.iloc[0]["policy"],
            "surrogate_best_second_gap": (
                float(surrogate_values[0] - surrogate_values[1])
                if len(surrogate_values) > 1
                else np.nan
            ),
            "table_best_second_gap": (
                float(table_values[0] - table_values[1])
                if len(table_values) > 1
                else np.nan
            ),
            "top_3_policy_overlap": len(
                set(ordered_surrogate.head(3)["policy"])
                & set(ordered_table.head(3)["policy"])
            )
            / min(3, len(group)),
        }
    )

ranking_diagnostics = pd.DataFrame(ranking_records)
display(
    ranking_diagnostics.groupby("species", as_index=False).agg(
        stands=("stand_id", "size"),
        median_spearman=("spearman_policy_rank_correlation", "median"),
        minimum_spearman=("spearman_policy_rank_correlation", "min"),
        mean_top_3_overlap=("top_3_policy_overlap", "mean"),
        median_surrogate_decision_margin=(
            "surrogate_best_second_gap",
            "median",
        ),
        median_table_decision_margin=("table_best_second_gap", "median"),
    )
)
ranking_diagnostics.to_csv(
    OUTPUT_DIRECTORY / "policy_ranking_diagnostics.csv", index=False
)

## 11. Exact analytical NPV solution and comparison of the binary variables \(x_{ij}\)

In [ ]:
def select_independent_npv_optimum(
    economic_table: pd.DataFrame,
    value_column: str,
) -> pd.DataFrame:
    """Select one maximum-NPV policy per stand with deterministic tie-breaking."""
    selected = (
        economic_table.sort_values(
            ["stand_id", value_column, "policy"],
            ascending=[True, False, True],
        )
        .groupby("stand_id", as_index=False)
        .head(1)
        .reset_index(drop=True)
    )
    return selected


surrogate_selected = select_independent_npv_optimum(
    npv_comparison,
    "surrogate_npv",
)
table_selected = select_independent_npv_optimum(
    npv_comparison,
    "table_npv",
)

selected_policy_comparison = surrogate_selected[
    ["stand_id", "species", "policy", "surrogate_npv", "table_npv"]
].rename(
    columns={
        "policy": "surrogate_selected_policy",
        "surrogate_npv": "surrogate_npv_of_surrogate_solution",
        "table_npv": "table_npv_of_surrogate_solution",
    }
).merge(
    table_selected[
        ["stand_id", "policy", "surrogate_npv", "table_npv"]
    ].rename(
        columns={
            "policy": "table_selected_policy",
            "surrogate_npv": "surrogate_npv_of_table_solution",
            "table_npv": "table_npv_of_table_solution",
        }
    ),
    on="stand_id",
    how="inner",
    validate="one_to_one",
)
selected_policy_comparison["same_policy"] = (
    selected_policy_comparison["surrogate_selected_policy"]
    == selected_policy_comparison["table_selected_policy"]
)
selected_policy_comparison["table_regret_from_surrogate_decision"] = (
    selected_policy_comparison["table_npv_of_table_solution"]
    - selected_policy_comparison["table_npv_of_surrogate_solution"]
)
selected_policy_comparison["surrogate_regret_from_table_decision"] = (
    selected_policy_comparison["surrogate_npv_of_surrogate_solution"]
    - selected_policy_comparison["surrogate_npv_of_table_solution"]
)

# Explicit binary x_ij table for all alternatives.
x_comparison = npv_comparison[
    ["stand_id", "species", "policy", "surrogate_npv", "table_npv"]
].copy()
surrogate_selected_keys = set(
    zip(surrogate_selected["stand_id"], surrogate_selected["policy"])
)
table_selected_keys = set(zip(table_selected["stand_id"], table_selected["policy"]))
x_comparison["x_surrogate"] = [
    int((stand_id, policy) in surrogate_selected_keys)
    for stand_id, policy in zip(x_comparison["stand_id"], x_comparison["policy"])
]
x_comparison["x_table"] = [
    int((stand_id, policy) in table_selected_keys)
    for stand_id, policy in zip(x_comparison["stand_id"], x_comparison["policy"])
]
x_comparison["x_absolute_difference"] = (
    x_comparison["x_surrogate"] - x_comparison["x_table"]
).abs()

stand_policy_agreement = float(selected_policy_comparison["same_policy"].mean())
binary_variable_agreement = float(
    x_comparison["x_surrogate"].eq(x_comparison["x_table"]).mean()
)
binary_hamming_distance = int(x_comparison["x_absolute_difference"].sum())

print("Stand-level policy agreement:", f"{stand_policy_agreement:.2%}")
print("Binary-variable agreement:", f"{binary_variable_agreement:.2%}")
print("Binary x_ij Hamming distance:", binary_hamming_distance)

display(
    selected_policy_comparison.groupby("species", as_index=False).agg(
        stands=("stand_id", "size"),
        same_policy_fraction=("same_policy", "mean"),
        mean_table_regret=("table_regret_from_surrogate_decision", "mean"),
        maximum_table_regret=("table_regret_from_surrogate_decision", "max"),
    )
)
display(selected_policy_comparison.loc[~selected_policy_comparison["same_policy"]].head(20))

selected_policy_comparison.to_csv(
    OUTPUT_DIRECTORY / "selected_policy_comparison.csv", index=False
)
x_comparison.to_csv(
    OUTPUT_DIRECTORY / "x_ij_binary_comparison.csv", index=False
)

## 12. Cross-evaluated objective regret

In [ ]:
surrogate_optimum_npv = float(
    selected_policy_comparison["surrogate_npv_of_surrogate_solution"].sum()
)
surrogate_value_of_table_solution = float(
    selected_policy_comparison["surrogate_npv_of_table_solution"].sum()
)
table_optimum_npv = float(
    selected_policy_comparison["table_npv_of_table_solution"].sum()
)
table_value_of_surrogate_solution = float(
    selected_policy_comparison["table_npv_of_surrogate_solution"].sum()
)

table_regret = table_optimum_npv - table_value_of_surrogate_solution
surrogate_regret = surrogate_optimum_npv - surrogate_value_of_table_solution

regret_summary = pd.DataFrame(
    [
        {
            "evaluation_representation": "original_tables",
            "optimal_solution": "table_solution",
            "optimal_value": table_optimum_npv,
            "cross_evaluated_solution": "surrogate_solution",
            "cross_evaluated_value": table_value_of_surrogate_solution,
            "absolute_regret": table_regret,
            "relative_regret_percent": (
                100.0 * table_regret / abs(table_optimum_npv)
                if abs(table_optimum_npv) > 1e-9
                else np.nan
            ),
        },
        {
            "evaluation_representation": "surrogate_functions",
            "optimal_solution": "surrogate_solution",
            "optimal_value": surrogate_optimum_npv,
            "cross_evaluated_solution": "table_solution",
            "cross_evaluated_value": surrogate_value_of_table_solution,
            "absolute_regret": surrogate_regret,
            "relative_regret_percent": (
                100.0 * surrogate_regret / abs(surrogate_optimum_npv)
                if abs(surrogate_optimum_npv) > 1e-9
                else np.nan
            ),
        },
    ]
)

display(regret_summary)
regret_summary.to_csv(
    OUTPUT_DIRECTORY / "cross_evaluated_regret.csv", index=False
)

## 13. Comparison of annual harvested biomass \(H_t\)

In [ ]:
def selected_trajectory_table(
    forest: Sequence[pd.DataFrame],
    selected_keys: set[tuple[str, str]],
) -> pd.DataFrame:
    long_table = pd.concat(forest, ignore_index=True)
    mask = [
        (stand_id, policy) in selected_keys
        for stand_id, policy in zip(long_table["stand_id"], long_table["policy"])
    ]
    return long_table.loc[mask].copy()


def aggregate_harvest_schedule(
    forest: Sequence[pd.DataFrame],
    selected_keys: set[tuple[str, str]],
    column_name: str,
) -> pd.DataFrame:
    selected = selected_trajectory_table(forest, selected_keys)
    return (
        selected.groupby("period", as_index=False)["harvested_dry_wood_t"]
        .sum()
        .rename(columns={"harvested_dry_wood_t": column_name})
    )


annual_harvest_comparison = (
    aggregate_harvest_schedule(
        surrogate_forest,
        surrogate_selected_keys,
        "surrogate_growth_surrogate_solution_Mg",
    )
    .merge(
        aggregate_harvest_schedule(
            table_forest,
            table_selected_keys,
            "table_growth_table_solution_Mg",
        ),
        on="period",
        validate="one_to_one",
    )
    .merge(
        aggregate_harvest_schedule(
            table_forest,
            surrogate_selected_keys,
            "table_growth_surrogate_solution_Mg",
        ),
        on="period",
        validate="one_to_one",
    )
    .merge(
        aggregate_harvest_schedule(
            surrogate_forest,
            table_selected_keys,
            "surrogate_growth_table_solution_Mg",
        ),
        on="period",
        validate="one_to_one",
    )
)

annual_harvest_comparison["main_schedule_difference_Mg"] = (
    annual_harvest_comparison["surrogate_growth_surrogate_solution_Mg"]
    - annual_harvest_comparison["table_growth_table_solution_Mg"]
)
annual_harvest_comparison["main_schedule_absolute_difference_Mg"] = (
    annual_harvest_comparison["main_schedule_difference_Mg"].abs()
)

schedule_error = annual_harvest_comparison["main_schedule_difference_Mg"].to_numpy()
annual_harvest_summary = pd.DataFrame(
    [
        {
            "surrogate_solution_total_harvest_Mg": annual_harvest_comparison[
                "surrogate_growth_surrogate_solution_Mg"
            ].sum(),
            "table_solution_total_harvest_Mg": annual_harvest_comparison[
                "table_growth_table_solution_Mg"
            ].sum(),
            "total_harvest_difference_Mg": schedule_error.sum(),
            "annual_harvest_rmse_Mg": float(np.sqrt(np.mean(schedule_error**2))),
            "annual_harvest_mae_Mg": float(np.mean(np.abs(schedule_error))),
            "maximum_annual_difference_Mg": float(np.max(np.abs(schedule_error))),
            "annual_schedule_correlation": float(
                np.corrcoef(
                    annual_harvest_comparison[
                        "surrogate_growth_surrogate_solution_Mg"
                    ],
                    annual_harvest_comparison[
                        "table_growth_table_solution_Mg"
                    ],
                )[0, 1]
            ),
        }
    ]
)

display(annual_harvest_summary)
display(annual_harvest_comparison)

annual_harvest_comparison.to_csv(
    OUTPUT_DIRECTORY / "annual_harvest_H_t_comparison.csv", index=False
)
annual_harvest_summary.to_csv(
    OUTPUT_DIRECTORY / "annual_harvest_summary.csv", index=False
)

In [ ]:
fig, axis = plt.subplots(figsize=(13, 6))
axis.plot(
    annual_harvest_comparison["period"],
    annual_harvest_comparison["surrogate_growth_surrogate_solution_Mg"],
    marker="o",
    label="Surrogate optimum evaluated with surrogate growth",
)
axis.plot(
    annual_harvest_comparison["period"],
    annual_harvest_comparison["table_growth_table_solution_Mg"],
    marker="s",
    label="Table optimum evaluated with table growth",
)
axis.set_xlabel("Period")
axis.set_ylabel("Harvested dry biomass (Mg)")
axis.set_title("Annual harvested-biomass variable $H_t$")
axis.grid(alpha=0.25)
axis.legend()
fig.tight_layout()
fig.savefig(
    OUTPUT_DIRECTORY / "annual_harvest_H_t_comparison.png",
    dpi=220,
    bbox_inches="tight",
)
plt.show()

fig, axis = plt.subplots(figsize=(13, 5))
axis.bar(
    annual_harvest_comparison["period"],
    annual_harvest_comparison["main_schedule_difference_Mg"],
)
axis.axhline(0.0, linewidth=1)
axis.set_xlabel("Period")
axis.set_ylabel("Surrogate minus table harvest (Mg)")
axis.set_title("Annual harvested-biomass differences")
axis.grid(axis="y", alpha=0.25)
fig.tight_layout()
fig.savefig(
    OUTPUT_DIRECTORY / "annual_harvest_H_t_differences.png",
    dpi=220,
    bbox_inches="tight",
)
plt.show()

## 14. Optional MILP verification with Treemün

The analytical solution above is the exact optimum of this benchmark because no constraint couples the decisions of different stands. This section nevertheless builds and solves both Treemün models to verify that:

- the extracted binary variables reproduce the analytical selections;
- the model variable `harvested_dry_wood_t[period]` (legacy API name; values are Mg) reproduces the aggregated \(H_t\) schedules.

In [ ]:
def detect_solver_thread_budget(cap: int = 8) -> int:
    scheduler_variables = (
        "SLURM_CPUS_PER_TASK",
        "SLURM_CPUS_ON_NODE",
        "PBS_NP",
        "NSLOTS",
    )
    for variable in scheduler_variables:
        raw_value = os.getenv(variable)
        if raw_value:
            try:
                return max(1, min(int(cap), int(raw_value)))
            except ValueError:
                pass
    return max(1, min(int(cap), int(os.cpu_count() or 1)))


def select_available_solver(preferred: str | None = None):
    try:
        import pyomo.environ as pyo
    except ImportError:
        return None, {}

    availability = {}
    for candidate in SOLVER_CANDIDATES:
        try:
            solver = pyo.SolverFactory(candidate)
            availability[candidate] = bool(
                solver is not None
                and solver.available(exception_flag=False)
            )
        except Exception:
            availability[candidate] = False

    if preferred and availability.get(preferred, False):
        return preferred, availability
    for candidate in SOLVER_CANDIDATES:
        if availability.get(candidate, False):
            return candidate, availability
    return None, availability


SOLVER_NAME, SOLVER_AVAILABILITY = select_available_solver(PREFERRED_SOLVER)
SOLVER_THREADS = detect_solver_thread_budget(SOLVER_THREAD_CAP)
SOLVER_OPTIONS = {
    "relative_gap": RELATIVE_GAP,
    "threads": SOLVER_THREADS,
    "time_limit_seconds": SOLVER_TIME_LIMIT_SECONDS,
    "tee": False,
}

print("Solver availability:", SOLVER_AVAILABILITY)
print("Selected solver:", SOLVER_NAME)
print("Requested solver threads:", SOLVER_THREADS)

In [ ]:
def solve_pure_npv_model(
    forest,
    policy_summary,
    ending_stock,
    harvested_by_period,
    *,
    solver_name: str,
):
    model = tm.build_forest_management_model(
        policy_summary=policy_summary,
        ending_dry_wood_by_policy=ending_stock,
        harvested_dry_wood_by_period=harvested_by_period,
        horizon=HORIZON,
        forest=forest,
        economic_mode="detailed_cash_flow",
        economic_parameters=CHILE_2025_NPV_PARAMETERS,
        discount_rate=DISCOUNT_RATE,
        minimum_ending_dry_wood_t=0.0,
        minimum_ending_dry_wood_fraction=None,
        even_flow_mode="none",
        objective="economic",
    )
    results = tm.solve_model(model, solver_name, **SOLVER_OPTIONS)
    return tm.extract_solution(model, results)


milp_verification = None
if RUN_MILP_VERIFICATION and SOLVER_NAME is not None:
    surrogate_milp_solution = solve_pure_npv_model(
        surrogate_forest,
        surrogate_policy_summary,
        surrogate_ending_stock,
        surrogate_harvested_by_period,
        solver_name=SOLVER_NAME,
    )
    table_milp_solution = solve_pure_npv_model(
        table_forest,
        table_policy_summary,
        table_ending_stock,
        table_harvested_by_period,
        solver_name=SOLVER_NAME,
    )

    surrogate_milp_keys = set(
        zip(
            surrogate_milp_solution["selected_policies"]["stand_id"],
            surrogate_milp_solution["selected_policies"]["policy"],
        )
    )
    table_milp_keys = set(
        zip(
            table_milp_solution["selected_policies"]["stand_id"],
            table_milp_solution["selected_policies"]["policy"],
        )
    )

    surrogate_x_match = surrogate_milp_keys == surrogate_selected_keys
    table_x_match = table_milp_keys == table_selected_keys

    surrogate_schedule_check = surrogate_milp_solution["harvest_schedule"].merge(
        annual_harvest_comparison[
            ["period", "surrogate_growth_surrogate_solution_Mg"]
        ],
        on="period",
        validate="one_to_one",
    )
    table_schedule_check = table_milp_solution["harvest_schedule"].merge(
        annual_harvest_comparison[
            ["period", "table_growth_table_solution_Mg"]
        ],
        on="period",
        validate="one_to_one",
    )

    surrogate_H_match = np.allclose(
        surrogate_schedule_check["harvested_dry_wood_t"],
        surrogate_schedule_check["surrogate_growth_surrogate_solution_Mg"],
        atol=1e-5,
    )
    table_H_match = np.allclose(
        table_schedule_check["harvested_dry_wood_t"],
        table_schedule_check["table_growth_table_solution_Mg"],
        atol=1e-5,
    )

    milp_verification = pd.DataFrame(
        [
            {
                "representation": "surrogate",
                "analytical_x_matches_milp": surrogate_x_match,
                "analytical_H_t_matches_milp": surrogate_H_match,
                "milp_npv": surrogate_milp_solution["net_present_value"],
                "solve_time_seconds": surrogate_milp_solution[
                    "solve_time_seconds"
                ],
            },
            {
                "representation": "original_tables",
                "analytical_x_matches_milp": table_x_match,
                "analytical_H_t_matches_milp": table_H_match,
                "milp_npv": table_milp_solution["net_present_value"],
                "solve_time_seconds": table_milp_solution[
                    "solve_time_seconds"
                ],
            },
        ]
    )
    display(milp_verification)
    milp_verification.to_csv(
        OUTPUT_DIRECTORY / "milp_verification.csv", index=False
    )
else:
    print(
        "MILP verification skipped. The analytical solution remains exact "
        "for this uncoupled pure-NPV benchmark."
    )

## 15. Optional ex post carbon diagnostic — not used in optimization

In [ ]:
carbon_diagnostic = None
if RUN_OPTIONAL_CARBON_DIAGNOSTIC:
    carbon_accounting = tm.StandingWoodCarbonAccounting()
    surrogate_carbon_forest = carbon_accounting.add_to_forest(surrogate_forest)
    table_carbon_forest = carbon_accounting.add_to_forest(table_forest)

    surrogate_selected_carbon = selected_trajectory_table(
        surrogate_carbon_forest,
        surrogate_selected_keys,
    )
    table_selected_carbon = selected_trajectory_table(
        table_carbon_forest,
        table_selected_keys,
    )

    carbon_diagnostic = pd.DataFrame(
        [
            {
                "representation": "surrogate",
                "selected_solution": "surrogate_solution",
                "carbon_stock_time_Mg_C_yr": surrogate_selected_carbon[
                    "carbon_stock_time_tC_year"
                ].sum(),
                "ending_carbon_Mg_C": surrogate_selected_carbon.loc[
                    surrogate_selected_carbon["period"].eq(HORIZON),
                    "standing_wood_carbon_tC_after_operation",
                ].sum(),
            },
            {
                "representation": "original_tables",
                "selected_solution": "table_solution",
                "carbon_stock_time_Mg_C_yr": table_selected_carbon[
                    "carbon_stock_time_tC_year"
                ].sum(),
                "ending_carbon_Mg_C": table_selected_carbon.loc[
                    table_selected_carbon["period"].eq(HORIZON),
                    "standing_wood_carbon_tC_after_operation",
                ].sum(),
            },
        ]
    )
    display(carbon_diagnostic)
    carbon_diagnostic.to_csv(
        OUTPUT_DIRECTORY / "optional_ex_post_carbon_diagnostic.csv",
        index=False,
    )
else:
    print(
        "Carbon diagnostic is disabled. Set "
        "RUN_OPTIONAL_CARBON_DIAGNOSTIC = True to run it."
    )

## 16. Consolidated benchmark summary

In [ ]:
benchmark_summary = pd.DataFrame(
    [
        {
            "number_of_stands": stands["stand_id"].nunique(),
            "number_of_common_alternatives": len(surrogate_keys),
            "curve_points_compared": len(curve_comparison),
            "stand_level_policy_agreement": stand_policy_agreement,
            "binary_x_ij_agreement": binary_variable_agreement,
            "binary_x_ij_hamming_distance": binary_hamming_distance,
            "surrogate_optimum_npv": surrogate_optimum_npv,
            "table_optimum_npv": table_optimum_npv,
            "table_value_of_surrogate_solution": table_value_of_surrogate_solution,
            "table_regret_from_surrogate_solution": table_regret,
            "table_relative_regret_percent": (
                100.0 * table_regret / abs(table_optimum_npv)
                if abs(table_optimum_npv) > 1e-9
                else np.nan
            ),
            "annual_harvest_rmse_Mg": annual_harvest_summary.iloc[0][
                "annual_harvest_rmse_Mg"
            ],
            "annual_harvest_mae_Mg": annual_harvest_summary.iloc[0][
                "annual_harvest_mae_Mg"
            ],
            "annual_schedule_correlation": annual_harvest_summary.iloc[0][
                "annual_schedule_correlation"
            ],
        }
    ]
)

display(benchmark_summary.T.rename(columns={0: "value"}))
benchmark_summary.to_csv(
    OUTPUT_DIRECTORY / "benchmark_summary.csv", index=False
)

print("\nInterpretation guide")
print("--------------------")
print(
    "1. Curve metrics measure numerical fit at the observations used to "
    "construct the surrogate functions."
)
print(
    "2. Policy agreement measures nominal decision identity, but regret is "
    "the stronger measure when several policies are nearly tied."
)
print(
    "3. The table-evaluated regret of the surrogate solution is the primary "
    "decision-fidelity statistic."
)
print(
    "4. H_t differences reveal whether small curve errors propagate into a "
    "materially different harvest schedule."
)
print("\nOutputs written to:", OUTPUT_DIRECTORY)

## 17. Final benchmark visualizations

This section consolidates the figures needed to close the benchmark.
The main comparison is the annual harvested dry-biomass variable
\(H_t\) obtained from the independently optimized surrogate and
source-table representations. Additional figures diagnose curve
fidelity, cumulative harvest, policy agreement, cross-evaluated NPV,
and policy transitions.

In [ ]:
FINAL_FIGURE_DIRECTORY = OUTPUT_DIRECTORY / "figures_final"
FINAL_FIGURE_DIRECTORY.mkdir(parents=True, exist_ok=True)

plt.rcParams.update(
    {
        "figure.dpi": 110,
        "savefig.dpi": 300,
        "axes.grid": True,
        "grid.alpha": 0.25,
        "font.size": 11,
    }
)

print("Final benchmark figures:", FINAL_FIGURE_DIRECTORY)

### 17.1 Annual harvested biomass under each optimized representation

In [ ]:
# Primary benchmark figure: each representation is optimized and
# evaluated using its own biomass engine under identical NPV settings.
fig, axis = plt.subplots(figsize=(13, 6.5))
axis.plot(
    annual_harvest_comparison["period"],
    annual_harvest_comparison[
        "surrogate_growth_surrogate_solution_Mg"
    ],
    marker="o",
    linewidth=2,
    label="Treemün surrogate functions",
)
axis.plot(
    annual_harvest_comparison["period"],
    annual_harvest_comparison[
        "table_growth_table_solution_Mg"
    ],
    marker="s",
    linewidth=2,
    label="Original source tables",
)
axis.set_xlabel("Planning period")
axis.set_ylabel("Harvested dry biomass (Mg)")
axis.set_title(
    "Annual harvested dry biomass under the two optimized models"
)
axis.legend()
fig.tight_layout()
for suffix in ("png", "pdf"):
    fig.savefig(
        FINAL_FIGURE_DIRECTORY
        / f"benchmark_annual_harvest_main.{suffix}",
        bbox_inches="tight",
    )
plt.show()

display(annual_harvest_summary)

### 17.2 Annual and cumulative harvest deviations

In [ ]:
fig, axis = plt.subplots(figsize=(13, 5.5))
axis.bar(
    annual_harvest_comparison["period"],
    annual_harvest_comparison["main_schedule_difference_Mg"],
)
axis.axhline(0.0, linewidth=1)
axis.set_xlabel("Planning period")
axis.set_ylabel("Surrogate minus table harvest (Mg)")
axis.set_title("Annual harvested-biomass deviation")
fig.tight_layout()
for suffix in ("png", "pdf"):
    fig.savefig(
        FINAL_FIGURE_DIRECTORY
        / f"benchmark_annual_harvest_difference.{suffix}",
        bbox_inches="tight",
    )
plt.show()

cumulative_harvest = annual_harvest_comparison[
    [
        "period",
        "surrogate_growth_surrogate_solution_Mg",
        "table_growth_table_solution_Mg",
    ]
].copy()
cumulative_harvest["surrogate_cumulative_Mg"] = cumulative_harvest[
    "surrogate_growth_surrogate_solution_Mg"
].cumsum()
cumulative_harvest["table_cumulative_Mg"] = cumulative_harvest[
    "table_growth_table_solution_Mg"
].cumsum()
cumulative_harvest["cumulative_difference_Mg"] = (
    cumulative_harvest["surrogate_cumulative_Mg"]
    - cumulative_harvest["table_cumulative_Mg"]
)

fig, axis = plt.subplots(figsize=(13, 6))
axis.plot(
    cumulative_harvest["period"],
    cumulative_harvest["surrogate_cumulative_Mg"],
    linewidth=2,
    label="Treemün surrogate functions",
)
axis.plot(
    cumulative_harvest["period"],
    cumulative_harvest["table_cumulative_Mg"],
    linewidth=2,
    label="Original source tables",
)
axis.set_xlabel("Planning period")
axis.set_ylabel("Cumulative harvested dry biomass (Mg)")
axis.set_title("Cumulative harvested dry biomass")
axis.legend()
fig.tight_layout()
for suffix in ("png", "pdf"):
    fig.savefig(
        FINAL_FIGURE_DIRECTORY
        / f"benchmark_cumulative_harvest.{suffix}",
        bbox_inches="tight",
    )
plt.show()

cumulative_harvest.to_csv(
    OUTPUT_DIRECTORY / "cumulative_harvest_comparison.csv",
    index=False,
)

### 17.3 Source-table biomass versus surrogate predictions

In [ ]:
minimum_biomass = float(
    min(
        curve_comparison["source_dry_biomass_Mg_ha"].min(),
        curve_comparison["surrogate_dry_biomass_Mg_ha"].min(),
    )
)
maximum_biomass = float(
    max(
        curve_comparison["source_dry_biomass_Mg_ha"].max(),
        curve_comparison["surrogate_dry_biomass_Mg_ha"].max(),
    )
)

fig, axis = plt.subplots(figsize=(7.5, 7.5))
for species, group in curve_comparison.groupby("species"):
    axis.scatter(
        group["source_dry_biomass_Mg_ha"],
        group["surrogate_dry_biomass_Mg_ha"],
        alpha=0.65,
        label=species,
    )
axis.plot(
    [minimum_biomass, maximum_biomass],
    [minimum_biomass, maximum_biomass],
    linestyle="--",
    linewidth=1.5,
    label="1:1 reference",
)
axis.set_xlabel("Original table biomass (Mg ha$^{-1}$)")
axis.set_ylabel("Surrogate prediction (Mg ha$^{-1}$)")
axis.set_title("Curve-level fidelity at source-table observations")
axis.set_aspect("equal", adjustable="box")
axis.legend()
fig.tight_layout()
for suffix in ("png", "pdf"):
    fig.savefig(
        FINAL_FIGURE_DIRECTORY
        / f"benchmark_curve_fidelity_scatter.{suffix}",
        bbox_inches="tight",
    )
plt.show()

### 17.4 Residual behavior across biological age

In [ ]:
fig, axis = plt.subplots(figsize=(11, 6))
for species, group in curve_comparison.groupby("species"):
    axis.scatter(
        group["age"],
        group["residual_Mg_ha"],
        alpha=0.65,
        label=species,
    )
axis.axhline(0.0, linewidth=1)
axis.set_xlabel("Biological age")
axis.set_ylabel("Surrogate minus source table (Mg ha$^{-1}$)")
axis.set_title("Surrogate residuals by biological age")
axis.legend()
fig.tight_layout()
for suffix in ("png", "pdf"):
    fig.savefig(
        FINAL_FIGURE_DIRECTORY
        / f"benchmark_residuals_by_age.{suffix}",
        bbox_inches="tight",
    )
plt.show()

### 17.5 Decision agreement and cross-evaluated regret

In [ ]:
agreement_by_species = (
    selected_policy_comparison.groupby("species", as_index=False)
    .agg(
        stands=("stand_id", "size"),
        policy_agreement=("same_policy", "mean"),
        mean_table_regret=(
            "table_regret_from_surrogate_decision",
            "mean",
        ),
    )
)

fig, axis = plt.subplots(figsize=(8, 5.5))
axis.bar(
    agreement_by_species["species"],
    100.0 * agreement_by_species["policy_agreement"],
)
axis.set_ylim(0.0, 100.0)
axis.set_ylabel("Identical selected policies (%)")
axis.set_title("Stand-level policy agreement by species")
fig.tight_layout()
for suffix in ("png", "pdf"):
    fig.savefig(
        FINAL_FIGURE_DIRECTORY
        / f"benchmark_policy_agreement_by_species.{suffix}",
        bbox_inches="tight",
    )
plt.show()

cross_evaluation_plot = pd.DataFrame(
    {
        "case": [
            "Table optimum\n(table evaluation)",
            "Surrogate solution\n(table evaluation)",
            "Surrogate optimum\n(surrogate evaluation)",
            "Table solution\n(surrogate evaluation)",
        ],
        "npv": [
            table_optimum_npv,
            table_value_of_surrogate_solution,
            surrogate_optimum_npv,
            surrogate_value_of_table_solution,
        ],
    }
)

fig, axis = plt.subplots(figsize=(10, 6))
axis.bar(
    cross_evaluation_plot["case"],
    cross_evaluation_plot["npv"] / 1_000_000.0,
)
axis.set_ylabel("NPV (million constant 2025 USD)")
axis.set_title("Cross-evaluated optimal and transferred decisions")
axis.tick_params(axis="x", rotation=12)
fig.tight_layout()
for suffix in ("png", "pdf"):
    fig.savefig(
        FINAL_FIGURE_DIRECTORY
        / f"benchmark_cross_evaluated_npv.{suffix}",
        bbox_inches="tight",
    )
plt.show()

display(agreement_by_species)
display(regret_summary)

### 17.6 Policy-transition matrix

In [ ]:
policy_transition = pd.crosstab(
    selected_policy_comparison["surrogate_selected_policy"],
    selected_policy_comparison["table_selected_policy"],
)
all_policies = sorted(
    set(policy_transition.index) | set(policy_transition.columns)
)
policy_transition = policy_transition.reindex(
    index=all_policies,
    columns=all_policies,
    fill_value=0,
)

fig, axis = plt.subplots(figsize=(10, 9))
image = axis.imshow(policy_transition.to_numpy(), aspect="auto")
axis.set_xticks(np.arange(len(all_policies)))
axis.set_yticks(np.arange(len(all_policies)))
axis.set_xticklabels(all_policies, rotation=90)
axis.set_yticklabels(all_policies)
axis.set_xlabel("Policy selected with original tables")
axis.set_ylabel("Policy selected with surrogate functions")
axis.set_title("Selected-policy transition matrix")
figure_colorbar = fig.colorbar(image, ax=axis)
figure_colorbar.set_label("Number of stands")
fig.tight_layout()
for suffix in ("png", "pdf"):
    fig.savefig(
        FINAL_FIGURE_DIRECTORY
        / f"benchmark_policy_transition_matrix.{suffix}",
        bbox_inches="tight",
    )
plt.show()

policy_transition.to_csv(
    OUTPUT_DIRECTORY / "selected_policy_transition_matrix.csv"
)

### Final interpretation

The annual harvest plot should be interpreted together with the
decision-regret statistics. Different policy identifiers or annual
harvest peaks do not necessarily imply a material loss of decision
quality. The primary decision-fidelity measure remains the NPV regret
obtained when the surrogate-derived solution is re-evaluated using
the original source tables.

## Recommended interpretation for a manuscript

The main conclusions should be reported in this order:

1. **Statistical fidelity:** pooled and equation-specific \(R^2\), RMSE, MAE, bias, MAPE, and maximum error.
2. **Decision agreement:** percentage of stands with the same selected policy and the binary \(x_{ij}\) Hamming distance.
3. **Decision regret:** loss of table-based NPV when the surrogate-derived solution is evaluated with the source tables.
4. **Schedule fidelity:** RMSE, MAE, correlation, and maximum annual difference for \(H_t\).
5. **Support sensitivity:** distinguish observed/interpolated harvest events from extrapolated initial-rotation resets.

A low nominal policy agreement is not necessarily problematic when cross-evaluated regret is negligible. Conversely, a high \(R^2\) is not sufficient evidence of decision fidelity if the table-based regret or annual harvest discrepancies are material.


## Scientific-notation audit

All manuscript-facing axes, captions, local metric columns, and exported
summary tables use Mg-based notation. Historical suffixes may remain
only where the notebook calls Treemün's backward-compatible API.


In [ ]:

publication_labels = [
    r"Original table biomass (Mg ha$^{-1}$)",
    r"Surrogate prediction (Mg ha$^{-1}$)",
    "Harvested dry biomass (Mg)",
    "Carbon stock-time (Mg C yr)",
]

for label in publication_labels:
    print("OK:", label)

print(
    "\nInternal Treemün API names remain unchanged where required; "
    "all scientific reporting uses Mg-based notation."
)


In [ ]:
pwd